---
title: "DRG Checks"

author: "Carlos Resurreccion"

date: "2025-04-03"
---


# Setup


## Parameters

Change which year_to_load to process in
`~/pids-drg-claims/data-cleaning/debug/cache/year_to_load.txt`

Change main GLOBAL (i.e. across all scripts) parameters in
`~/pids-drg-claims/data-cleaning/00a-parameters.r`

Change seldom touched parameters in
`~/pids-drg-claims/data-cleaning/r_scripts_v2/0.1.0.params_fpaths.R`


In [ ]:
source(here::here("data-cleaning", "00a-parameters.r"))


## Libraries


In [ ]:
source(here::here("data-cleaning", "00b-packages.r"))


## R Scripts


In [ ]:
source(here::here("data-cleaning", "00c-load-params-and-scripts.r"))


## Load Mapping Data


In [ ]:
source(here::here("data-cleaning", "00d-load-mapping.r"))


## Load final .rds


In [44]:
dt_clean_bq_subset <- readRDS(here(
  chkpt_2_path,
  paste0(
    chkpt_2_prefix, year_to_load, suffix,
    "v2", "_part_b_bq_subset", ".rds"
  )
))
dt_clean_full <- readRDS(here(
  chkpt_2_path,
  paste0(
    chkpt_2_prefix, year_to_load, suffix,
    "v2", "_part_a_is_covid", ".rds"
  )
))


In [45]:
str(dt_clean_full)


Classes ‘data.table’ and 'data.frame':	12757064 obs. of  42 variables:
 $ id_year          : int  2022 2022 2022 2022 2022 2022 2022 2022 2022 2022 ...
 $ id_series        : chr  "4003684003031103203004" "1101201101170502211011" "2102012102020802202012" "3007153007121103217003" ...
 $ id_pin           : chr  "D09832BA95F246B62FD419737E1CE07F" "3CB620F99E50627B1D201165D9F67674" "CCD2898B64D33AADBEDA0B37AB64DC18" "F28AFC768E17CCED7901F45A8EE9E10D" ...
 $ id_hci           : chr  "300701" "590101" "280402" "Z04303" ...
 $ id_hcp           :List of 12757064
  ..$ : chr "26350"
  ..$ : chr "39901"
  ..$ : chr "19260"
  ..$ : chr "40623"
  ..$ : chr  "28790" "35549"
  ..$ : chr "37770"
  ..$ : chr "61814"
  ..$ : chr "27711"
  ..$ : chr "72353"
  ..$ : chr "23394"
  ..$ : chr "33316"
  ..$ : chr "71761"
  ..$ : chr "55296"
  ..$ : chr "27510"
  ..$ : chr "38843"
  ..$ : chr "11203"
  ..$ : chr  "69925" "73299"
  ..$ : chr "71507"
  ..$ : chr "27487"
  ..$ : chr "62340"
  ..$ : chr "59695"
  .

## Load raw file


In [ ]:
dt_raw <- fread(
  file = here(
    raw_claims_path,
    paste0(full_claims_prefix, year_to_load, file_type)
  ), colClasses = "character", header = TRUE,
  encoding = "Latin-1", na.strings = na_values
)

# Drop columns
cols_to_drop <- intersect(colnames(dt_raw), c(drop_cols, drop_cols_manual))
if (length(cols_to_drop) > 0) {
  dt_raw <- dt_raw[, (cols_to_drop) := NULL]
}

avail_cols <- colnames(dt_raw)

# Rename Columns
setnames(dt_raw,
  old = avail_cols[avail_cols %in% names(column_mappings)],
  new = unlist(column_mappings[
    avail_cols[
      avail_cols %in% names(column_mappings)
    ]
  ])
)

dt_raw[dt_raw == ""] <- NA_character_
dt_raw[, id_series := trimws(id_series)]


## Function Definitions


In [ ]:
test_checks <- function(section_id) {
  # Coerce to two-character string (e.g., 1 → "01")
  section_id <- sprintf("%02d", as.integer(section_id))

  # Get the calling environment (e.g., global or wherever this is invoked from)
  calling_env <- parent.frame()

  # Build pattern to match only variables for the given section
  pattern <- paste0("^chk_", section_id, "_\\d{2}_.+")

  # List relevant check variables in the calling environment
  chk_vars <- ls(envir = calling_env, pattern = pattern)

  # If no matching checks found, warn and exit
  if (length(chk_vars) == 0) {
    cat("⚠️ No checks found for section: ", section_id)
    return(invisible(NULL))
  }

  # Get values (assumed format: c(flag, info))
  chk_values_raw <- lapply(chk_vars, get, envir = calling_env)

  # Extract just the logical flag from each
  chk_flags <- sapply(chk_values_raw, function(x) isTRUE(x[1]))

  # Check for failures
  if (any(!chk_flags)) {
    failed_checks <- chk_vars[!chk_flags]

    # Build detailed failure messages
    failure_messages <- mapply(function(var, val) {
      suffix <- sub(paste0("^chk_", section_id, "_\\d{2}_(.+)$"), "\\1", var)
      info <- if (length(val) > 1) val[2] else "No additional info"
      paste0("• ", suffix, ": ", info)
    }, var = failed_checks, val = chk_values_raw[!chk_flags], SIMPLIFY = TRUE)

    stop(paste0(
      "❌ Validation failed in section ", section_id, ":\n",
      paste(failure_messages, collapse = "\n")
    ))
  } else {
    # Print all check results
    cat(paste0(
      "✅ All validation checks passed for section ",
      section_id, ":\n"
    ))
    for (i in seq_along(chk_vars)) {
      suffix <- sub(paste0(
        "^chk_", section_id,
        "_\\d{2}_(.+)$"
      ), "\\1", chk_vars[i])
      cat(paste0(suffix, ": ", chk_values_raw[[i]][1], "\n"))
    }
  }
}

custom_setdiff <- function(x, y) {
  # If equal, return TRUE with no message
  if (setequal(x, y)) {
    return(c(TRUE, NULL))
  } else {
    # Identify elements missing and extra
    missing_in_y <- setdiff(x, y) # present in x but missing in y
    extra_in_y <- setdiff(y, x) # present in y but not in x

    # Compose detailed message
    mismatch_msg <- paste0(
      if (length(missing_in_y)) {
        paste0("\n  - missing: ", paste(missing_in_y, collapse = ", "))
      } else {
        ""
      },
      if (length(extra_in_y)) {
        paste0("\n  - invalid: ", paste(extra_in_y, collapse = ", "))
      } else {
        ""
      }
    )

    return(c(FALSE, mismatch_msg))
  }
}

custom_setequal <- function(x, y) {
  # If equal, return TRUE with no message
  if (setequal(x, y)) {
    return(c(TRUE, NULL))
  } else {
    mismatch_msg <- paste0(
      "X: ", x, " Y: ", y
    )
    return(c(FALSE, mismatch_msg))
  }
}

custom_check_mappings <- function(result_list) {
  # Keep only the FALSE mappings
  filtered <- lapply(result_list, function(x) Filter(Negate(isTRUE), x))
  filtered <- Filter(length, filtered) # drop empty lists

  if (length(filtered) == 0) {
    return(c(TRUE, NULL))
  } else {
    ticker <- 0
    for (col in names(filtered)) {
      msg_lines <- if (ticker == 0) {
        c(paste0("\n $ ", col, ":"))
      } else {
        c(msg_lines, paste0(" $ ", col, ":"))
      }
      ticker <- ticker + 1
      for (raw_val in names(filtered[[col]])) {
        # Extract the actual wrongly mapped value from the original result_list
        wrong_val <- result_list[[col]][[raw_val]]
        msg_lines <- c(
          msg_lines,
          paste0('   $ "', raw_val, '": MAPPING FAILED')
        )
      }
    }
    return(c(FALSE, paste(msg_lines, collapse = "\n")))
  }
}


# Data Verification Proper


### Test Batch 01:


In [ ]:
cols_expected <- bq_cols
cols_actual <- colnames(dt_clean_bq_subset)
cols_schema <- fromJSON(here(
  "data-cleaning/r_scripts_v2",
  "bq_schema_cleaning.json"
))$name
chk_01_01_cols_match_expected <- custom_setdiff(cols_expected, cols_actual)
chk_01_02_cols_match_schema <- custom_setdiff(cols_schema, cols_actual)
test_checks(1)


### Test Batch 02:


In [ ]:
nrow_expected <- dt_raw[, .N]
nrow_actual_full <- nrow(dt_clean_bq_subset)
chk_02_01_nrows_match_full <- custom_setequal(
  nrow_expected, nrow_actual_full
)

# Function to check if MD5 hashes have changed, returning TRUE if no change
check_md5_changes <- function(year_to_load) {
  # Function to calculate and save MD5 hash for a given file
  calculate_md5 <- function(file_path) {
    md5sum <- digest::digest(file_path, algo = "md5", file = TRUE)
    return(md5sum)
  }
  hash_cache_dir <- here::here("data-cleaning/debug/cache/partial_md5")
  dir.create(hash_cache_dir, recursive = TRUE, showWarnings = FALSE)
  hash_file_path <- here::here(
    hash_cache_dir,
    paste0("md5_hashes_", year_to_load, ".rds")
  )

  # Generate new MD5 hashes for each part
  current_hashes <- sapply(1:split_parts, function(part) {
    part_file <- here::here(
      raw_claims_parts_path,
      paste0(
        full_claims_prefix, year_to_load, "_part_",
        sprintf("%02d", part), "_of_", split_parts, ".rds"
      )
    )
    calculate_md5(part_file)
  })

  # Check if saved hashes exist
  if (file.exists(hash_file_path)) {
    saved_hashes <- readRDS(hash_file_path)
    # Return TRUE if hashes match, indicating no changes
    if (identical(saved_hashes, current_hashes)) {
      cat(paste(
        "✅ No changes in partial files for eclaims year",
        year_to_load, "\n"
      ))
      return(TRUE)
    }
  }
  return(FALSE)
}
nrow_actual_partial <- nrow_partial <- 0
if (!check_md5_changes(year_to_load)) {
  for (loop_part in 1:split_parts) {
    nrow_partial <- nrow(read_appropriate_file(loop_part))
    nrow_actual_partial <- nrow_actual_partial + nrow_partial
  }
  chk_02_02_nrows_match_partial <- custom_setequal(
    nrow_expected, nrow_actual_partial
  )
} else {
  chk_02_02_nrows_match_partial <- c(TRUE, NULL)
}

test_checks(2)


### Test Batch 03:


In [ ]:
id_series_rows <- dt_clean_bq_subset[grepl("e", id_series), .(id_series)]
id_pin_rows <- dt_clean_bq_subset[grepl("e", id_pin), .(id_series, id_pin)]
id_hci_rows <- dt_clean_bq_subset[grepl("e", id_hci), .(id_series, id_hci)]
id_series_expo <-
  if (!is.null(id_series_rows)) {
    nrow(id_series_rows)
  } else {
    0
  }
id_pin_expo <-
  if (!is.null(id_pin_rows)) {
    nrow(id_pin_rows)
  } else {
    0
  }
id_hci_expo <-
  if (!is.null(id_hci_rows)) {
    nrow(id_hci_rows)
  } else {
    0
  }
chk_03_01_id_series_expo <-
  custom_setequal(id_series_expo, 0)
chk_03_02_id_pin_expo <-
  custom_setequal(id_pin_expo, 0)
chk_03_03_id_hci_expo <-
  custom_setequal(id_hci_expo, 0)
test_checks(3)


### Test Batch 04:


In [ ]:
mapping_results <- list()
if (to_debug) {
  cat("==================================================\n")
  flush.console()
}
# Subfunction: check mapping for one column
process_column_mapping <- function(
    col_name, dt_raw, dt_clean_bq_subset,
    expected_mappings, to_debug = FALSE) {
  if (to_debug) {
    cat(col_name, "\n--------------------------------------------------\n")
    flush.console()
  }
  result <- list()
  raw_col <- dt_raw[[col_name]]
  clean_col <- dt_clean_bq_subset[[col_name]]
  unique_raw_vals <- unique(na.omit(raw_col))
  for (raw_val in unique_raw_vals) {
    if (to_debug) {
      cat(raw_val, "\n")
      flush.console()
    }
    id_subset <- dt_raw[raw_col == raw_val, id_series]
    clean_subset <- unlist(
      dt_clean_bq_subset[id_series %chin% id_subset, .SD, .SDcols = col_name],
      use.names = FALSE
    )
    unique_clean_vals <- unique(na.omit(clean_subset))
    result_flag <-
      all(is.na(unique_clean_vals)) | length(unique_clean_vals) == 1
    result[[raw_val]] <- result_flag
    if (to_debug) {
      cat(str(id_subset), str(clean_subset), str(unique_clean_vals), sep = "\n")
      cat(result_flag, "\n--------------------------------------------------\n")
      flush.console()
    }
  }
  return(result)
}

mapping_results <- list()
if (to_debug) cat("==================================================\n")
if (to_debug) flush.console()
for (col_name in names(expected_mappings)) {
  mapping_results[[col_name]] <-
    process_column_mapping(
      col_name, dt_raw, dt_clean_bq_subset,
      expected_mappings, to_debug
    )
  if (to_debug) cat("==================================================\n")
  if (to_debug) flush.console()
}

if (to_debug) {
  str(mapping_results)
}

chk_04_01_mapping_results <- custom_check_mappings(mapping_results)
test_checks(04)


## Test Batch 05:


In [ ]:
dt_unacceptable_pdx <- dt_clean_bq_subset[
  !is.na(clin_pdx) &
    !clin_pdx %chin% acc_pdx,
  .(id_series, clin_pdx)
]
chk_05_01_no_unacceptable_pdx <-
  if (nrow(dt_unacceptable_pdx) == 0) {
    c(TRUE, NULL)
  } else {
    c(FALSE, paste(capture.output(str(dt_unacceptable_pdx)), collapse = "\n"))
  }
test_checks(5)


## Test Batch 06:
